In [ ]:
import spacy
from neo4j import GraphDatabase
import os


In [ ]:
NEO4J_URI = os.getenv("NEO4J_URI","neo4j://127.0.0.1:7687")
NEO4J_USER = os.getenv("NEO4J_USER","neo4j")
NEO4J_PW = os.getenv("NEO4J_PASSWORD","Concordia06")
driver = GraphDatabase.driver(
    "neo4j://127.0.0.1:7687",
    auth=("neo4j","Concordia06")
)
driver.verify_connectivity()

In [ ]:
a = os.getenv("NEO4J_PASSWORD")
print(a)

In [ ]:
with driver.session(database="neo4j") as session:
    results = session.run(
        """
        Match(n:WaterKgV2)
        RETURN
            elementId(n) AS element_id,
            labels(n) AS labels,
            properties(n) AS properties
        """
    )
    entity_list = set()
    property_list = set()
    property_value_list = set()
    for result in results:
        entity_list.update(result["labels"])
        property_list.update(result["properties"].keys())
        for value in result["properties"].values():
            if isinstance(value,str):
                property_value_list.add(value)


In [ ]:
entity_list

In [ ]:
property_list

In [ ]:
nlp = spacy.load("de_core_news_lg")

ruler = nlp.add_pipe(
    "entity_ruler",
    before="ner",
    config={"overwrite_ents",True}
    )


doc = nlp("Welche Phosphorwerte wurden im Müggelsee gemessen?")

for ent in doc.ents:
    print(ent.text, ent.label_)



In [ ]:
from lexicon_v2 import lookup, property_owners, build_alias_index

In [ ]:
lookup("monitoring_program")

In [ ]:
lookup("see")

In [ ]:
property_owners()

In [ ]:
import spacy
from spacy.matcher import PhraseMatcher

In [ ]:
nlp = spacy.load("de_core_news_lg")

In [ ]:
def build_spacy_alias_matcher(nlp) -> PhraseMatcher:
    alias_index = build_alias_index()

    matcher = PhraseMatcher(
        vocab=nlp.vocab,
        attr="LEMMA"
    )

    match_id_to_alias : dict[int, str] = {}

    for number,alias in enumerate(alias_index):
        rule_name = f"KG_ALIAS_{number}"

        pattern = nlp(alias)
        matcher.add(rule_name, [pattern])

        match_id = nlp.vocab.strings[rule_name]
        match_id_to_alias[match_id] = alias
    return matcher, match_id_to_alias


In [ ]:
matcher, match_id_to_alias = build_spacy_alias_matcher(nlp)

In [ ]:
match_id_to_alias

In [ ]:
alias_index = build_alias_index()

In [ ]:
def match_question(q: str) -> list[dict]:
    doc = nlp(q)
    matches = matcher(doc)

    assignments = []

    for match_id, start, end in matches:
        span = doc[start:end]
        alias = match_id_to_alias[match_id]
        assignments.append(
            {
                "text":span.text,
                "lemma": " ".join(
                token.lemma_
                for token in span
                ),
                "alias":alias,
                "start":start,
                "end":end,
                "candidates":alias_index[alias]
            }
        )
    return assignments

In [ ]:
match_question("Wie groß ist der Müggelsee?")

In [ ]:
from load_entities import load_entity_record
v = load_entity_record()

In [ ]:
set(
    a
    for b in v
    for a in b["alias_properties"]
)

In [ ]:
#todo : entity ruler
ruler = nlp.add_pipe(
    "entity_ruler",
    before="ner",
    config={
        "overwrite_ents": True,
    },
)

ruler.add_patterns(
    [
        {
            "label": "WATER_BODY",
            "pattern": "Müggelsee",
            "id": "water_body:123",
        },
        {
            "label": "PARAMETER",
            "pattern": "Gesamtphosphor",
            "id": "parameter:42",
        },
    ]
)